<a href="https://colab.research.google.com/github/ANNYM25/python-together/blob/main/09-fintech/05_credit_and_fintech_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5 · Credit scoring — underwriting as machine learning

This lesson ties the whole lab together: fintech underwriting ("will this
borrower default?") is a **classification** problem (ML track) on **structured**
financial data, where being wrong has a real cost. We'll generate synthetic
borrowers, fit a model, and judge it with the right metric. (Synthetic data,
illustrative only — real credit models face heavy fairness/regulatory scrutiny.)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split

## Generate synthetic borrowers

Three features: income, credit utilisation, and late payments in the last 12
months. We bake in a believable relationship (high utilisation and lates raise
default risk; higher income lowers it), then sample who actually defaulted.

In [ ]:
rng = np.random.default_rng(0)
n = 600
income = rng.lognormal(mean=10.5, sigma=0.4, size=n)
utilization = rng.uniform(0.05, 0.95, size=n)
late_12m = rng.poisson(0.4, size=n)

logit = -2.2 + 2.4 * utilization + 0.7 * late_12m - 0.00004 * income
p_default = 1 / (1 + np.exp(-logit))
y = rng.binomial(1, p_default)                      # 1 = defaulted
X = np.column_stack([income, utilization, late_12m])

print("overall default rate:", round(float(y.mean()), 3))

## Fit a logistic regression and score it with AUC

Defaults are rare, so (lesson 3!) accuracy is misleading. **AUC** measures how
well the model *ranks* risky borrowers above safe ones — the standard credit
metric.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)
model = LogisticRegression(max_iter=400).fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]

print("AUC:", round(roc_auc_score(y_test, proba), 3))
print("coefficients [income, utilisation, lates]:", model.coef_.round(4))
print("(utilisation & lates positive = raise risk; income negative = lowers it)")

## The decision: a threshold with a cost

A probability isn't a decision. You pick a cutoff to **decline** a loan — and
that choice trades off approving bad loans (default losses) against rejecting
good customers (lost business). Here's the confusion matrix at a strict cutoff.

In [ ]:
threshold = 0.25
decline = (proba >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, decline).ravel()
print(f"at threshold {threshold}: declined {decline.sum()} of {len(decline)} applicants")
print(f"  caught defaulters (TP): {tp},  missed (FN): {fn}")
print(f"  wrongly declined good (FP): {fp},  correctly approved (TN): {tn}")

## 🏋️ Your turn

Raise the `threshold` to `0.5` and compare: how many borrowers are declined, and
how many true defaulters slip through (FN)? This is the risk/growth dial a
lender actually turns.

In [ ]:
# threshold = 0.5
# decline = (proba >= threshold).astype(int)
# print("declined:", decline.sum())

### ✅ What you learned
- Credit underwriting is **classification on structured data** with real costs.
- Rare defaults mean **AUC**, not accuracy, is the right score (ranking quality).
- Coefficients show *direction* of each feature's effect.
- The decision **threshold** trades default losses against lost good customers.

🎉 **That's the fintech track** — and it uses every earlier track: money math,
stats, probability, and ML. Next stop: `10-projects/`.